# Cloud Computing (CS3081) Mini Project: Distilled Command Parser
## Complete Training Pipeline in Google Colab (Free T4 GPU)

This notebook runs the heavy compute steps of the assignment in **~12–15 minutes** on free GPU:
1. **Teacher Labelling**: Labels 3,000 commands with `microsoft/Phi-3.5-mini-instruct` (temperature 0, seed 42).
2. **Sharding**: Partitions into 4 shards (`shard-0.jsonl` through `shard-3.jsonl`).
3. **Data Cleaning (`clean.py`)**: Filters rules (a)–(e), checks the counting invariant, generates `datasets/v1/train.jsonl` and `train.sha256`.
4. **Student Distillation (`pythia-70m`)**: Fine-tunes `models/v1` and `models/v2` and exports `loss.csv`.
5. **Three-Way Evaluation (`score.py`)**: Evaluates Teacher, Untrained Base, and Student v1 & v2 on `validation.jsonl`.
6. **Release Packaging & Download**: Packages release tarballs and automatically downloads `cloud_assignment_artifacts.zip` to your computer.

---

### Step 1: Check GPU Accelerator & Install Libraries
*Make sure you selected **Runtime $\to$ Change runtime type $\to$ T4 GPU**.*

In [ ]:
!nvidia-smi
!pip install -q transformers torch accelerate datasets pyyaml boto3

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

### Step 2: Upload Project Files (`colab_starter.zip`)
Run this cell and click **Choose Files** to upload `colab_starter.zip` from your computer (located at `/Users/birampraveen/Downloads/cloud_assignment/colab_starter.zip`, ~80 KB).

In [ ]:
import os

from google.colab import files

if not os.path.exists("data/commands_pool.jsonl"):
    print("Please upload 'colab_starter.zip':")
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith(".zip"):
            !unzip -o "{fname}"
            print(f"Extracted {fname} successfully!")
else:
    print("Project data already present in working directory!")

!ls -la

### Step 3: Run the Complete Training & Labelling Pipeline
This runs the batched teacher labelling on GPU, data cleaning, fine-tuning Pythia-70m, multi-model evaluation, and packaging.

In [ ]:
!python scripts/kaggle_pipeline.py

### Step 4: Download Generated Artifacts to Your Computer
This triggers the browser download of `cloud_assignment_artifacts.zip` containing all trained weights, evaluation logs, and release tarballs.

In [ ]:
import os

from google.colab import files

zip_name = "cloud_assignment_artifacts.zip"
if os.path.exists(zip_name):
    size_mb = os.path.getsize(zip_name) / (1024 * 1024)
    print(f"Downloading {zip_name} ({size_mb:.1f} MB) to your local machine...")
    files.download(zip_name)
else:
    print(f"Error: {zip_name} does not exist. Please check the logs in Step 3 for any errors.")